# Project 1 - Image Viewer

This notebook demonstrates zoom, rotation, and cropping using the same reusable source code as the desktop application.

## 1. Objective

Study basic geometric image operations, compare interpolation methods, and measure processing time on sample images.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from apps.image_viewer.processing.crop import apply_crop
from apps.image_viewer.processing.rotate import apply_rotation
from apps.image_viewer.processing.zoom import apply_zoom
from shared.image_io import list_images, load_image
from shared.image_utils import bgr_to_rgb

DATASET = Path('datasets/project_1/input')
image_paths = list_images(DATASET)
print(f'Found {len(image_paths)} images in {DATASET}')
image_paths[:5]

## 2. Dataset

The assignment expects several local input images placed in `datasets/project_1/input/`. If the folder is empty, the code cell below will stop gracefully.

In [ ]:
if not image_paths:
    raise SystemExit('Please place sample images into datasets/project_1/input before running the notebook.')

sample_image = load_image(image_paths[0])
plt.figure(figsize=(8, 5))
plt.imshow(bgr_to_rgb(sample_image))
plt.title(f'Sample image: {image_paths[0].name}')
plt.axis('off');

## 3. Zoom

Image scaling resamples pixels onto a new coordinate grid. Different interpolation methods trade speed against smoothness.

In [ ]:
zoom_results = {
    name: apply_zoom(sample_image, 1.5, name)
    for name in ['Nearest Neighbor', 'Bilinear', 'Bicubic']
}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(bgr_to_rgb(sample_image))
axes[0].set_title('Original')
axes[0].axis('off')

for axis, (name, result) in zip(axes[1:], zoom_results.items()):
    axis.imshow(bgr_to_rgb(result.image))
    axis.set_title(f'{name}\n{result.processing_time_ms:.2f} ms')
    axis.axis('off')

plt.tight_layout();

## 4. Rotate

Rotation is an affine transformation. The image is rotated about its center using a 2x3 matrix produced by `cv2.getRotationMatrix2D`.

In [ ]:
angles = [30, 90, -45]
rotation_results = [apply_rotation(sample_image, angle, 'Bilinear', 'Reflect') for angle in angles]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(bgr_to_rgb(sample_image))
axes[0].set_title('Original')
axes[0].axis('off')

for axis, angle, result in zip(axes[1:], angles, rotation_results):
    axis.imshow(bgr_to_rgb(result.image))
    axis.set_title(f'{angle}°')
    axis.axis('off')

plt.tight_layout()
print('Rotation matrix for 30°:')
print(rotation_results[0].matrix)

## 5. Crop

Cropping uses NumPy slicing with the form `image[y1:y2, x1:x2]`.

In [ ]:
height, width = sample_image.shape[:2]
crop_result = apply_crop(sample_image, width // 6, height // 6, width // 2, height // 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(bgr_to_rgb(sample_image))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(bgr_to_rgb(crop_result.image))
axes[1].set_title(f'Cropped ({crop_result.retained_percentage:.1f}% retained)')
axes[1].axis('off')
plt.tight_layout();

## 6. Experimental Results

The table below evaluates several images with the same set of operations.

In [ ]:
rows = []
for path in image_paths[:5]:
    image = load_image(path)
    zoom = apply_zoom(image, 1.25, 'Bicubic')
    rotate = apply_rotation(image, 30, 'Bilinear', 'Reflect')
    crop = apply_crop(image, 0, 0, image.shape[1] // 2, image.shape[0] // 2)
    rows.extend([
        {'Image': path.name, 'Operation': 'Zoom', 'Parameters': '1.25x Bicubic', 'Processing time (ms)': zoom.processing_time_ms, 'Output size': zoom.output_size},
        {'Image': path.name, 'Operation': 'Rotate', 'Parameters': '30° Bilinear', 'Processing time (ms)': rotate.processing_time_ms, 'Output size': rotate.output_size},
        {'Image': path.name, 'Operation': 'Crop', 'Parameters': 'Top-left half', 'Processing time (ms)': crop.processing_time_ms, 'Output size': crop.image.shape[1::-1]},
    ])

display(HTML('<table border="1"><tr><th>Image</th><th>Operation</th><th>Parameters</th><th>Processing time (ms)</th><th>Output size</th></tr>' + ''.join(
    f"<tr><td>{row['Image']}</td><td>{row['Operation']}</td><td>{row['Parameters']}</td><td>{row['Processing time (ms)']:.2f}</td><td>{row['Output size']}</td></tr>" for row in rows
) + '</table>'))

## 7. Analysis

- Nearest neighbor is fastest but can create jagged edges.
- Bilinear often provides a balanced result.
- Bicubic is smoother but costs more time.
- Rotation can clip image corners when the canvas size is fixed.
- Cropping removes context but is computationally inexpensive.

## 8. Conclusion

Project 1 demonstrates how basic geometric image transformations behave visually and computationally while reusing the same source code as the PySide6 GUI.